## Merge and Finalize

In [1]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd
import numpy as np
from src.data_utils import load_raw, load_interim, save_processed


### Load

In [3]:
df = load_raw('application_train.csv')
df_bureau = load_interim('bureau_agg.csv')
df_prev = load_interim('previous_application_agg.csv')
df_inst = load_interim('installments_agg.csv')

### Merge All Aggregates

In [4]:
df_final = (df
            .merge(df_bureau, on='SK_ID_CURR', how='left')
            .merge(df_prev, on='SK_ID_CURR', how='left')
            .merge(df_inst, on='SK_ID_CURR', how='left')
           )

### Handle Missing Values

In [5]:
cat_col = df_final.select_dtypes(include='object').columns
df_final[cat_col] = df_final[cat_col].fillna('Unknown')

num_col = df_final.select_dtypes(exclude='object').columns
num_col = num_col.drop('TARGET')
df_final[num_col] = df_final[num_col].fillna(0)

### Light Percentile Capping

In [13]:
cap_col = [
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'INST_MAX_DELAY',
    'INST_AVG_DELAY'
]
for col in cap_col:
    lower = df_final[col].quantile(0.01)
    upper = df_final[col].quantile(0.99)
    df_final[col] = df_final[col].clip(lower, upper)

### Downcaste Datatype

In [14]:
for col in df_final.columns:
    if df_final[col].dtype == 'float64':
        df_final[col] = pd.to_numeric(df_final[col], downcast='float')
        
    if df_final[col].dtype == 'int64':
        df_final[col] = pd.to_numeric(df_final[col], downcast='integer')

### Save Final Dataset

In [15]:
save_processed(df_final, 'model_ready.csv')